In [ ]:
datasets = [
    {"folder":"texts/MAE", "ai":"human", "authors":"multiple authors", "lang":"estonian author"},
    {"folder":"texts/SAE", "ai":"human", "authors":"single author", "lang":"estonian author"},
    {"folder":"texts/MAU", "ai":"human",    "authors":"multiple authors", "lang":"british author"},
    {"folder":"texts/SAU", "ai":"human",    "authors":"single author", "lang":"british author"},
    {"folder":"ka", "ai":"human", "authors":"single author", "lang":"estonian author"},
    {"folder":"llmgenerated/SAE_gemini", "ai":"ai",    "authors":"single author", "lang":"estonian author"},
    {"folder":"llmgenerated/SAU_gemini", "ai":"ai",    "authors":"single author", "lang":"british author"},
    {"folder":"llmgenerated/MAE_gemini", "ai":"ai",    "authors":"multiple authors", "lang":"estonian author"},
    {"folder":"llmgenerated/MAU_gemini", "ai":"ai",    "authors":"multiple authors", "lang":"british author"},
    
    {"folder":"llmgenerated/SAE_deepseek", "ai":"ai",    "authors":"single author", "lang":"estonian author"},
    {"folder":"llmgenerated/SAU_deepseek", "ai":"ai",    "authors":"single author", "lang":"british author"},
    {"folder":"llmgenerated/MAE_deepseek", "ai":"ai",    "authors":"multiple authors", "lang":"estonian author"},
    {"folder":"llmgenerated/MAU_deepseek", "ai":"ai",    "authors":"multiple authors", "lang":"british author"},
    
    {"folder":"llmgenerated/SAE_gpt", "ai":"ai",    "authors":"single author", "lang":"estonian author"},
    {"folder":"llmgenerated/SAU_gpt", "ai":"ai",    "authors":"single author", "lang":"british author"},
    {"folder":"llmgenerated/MAE_gpt", "ai":"ai",    "authors":"multiple authors", "lang":"estonian author"},
    {"folder":"llmgenerated/MAU_gpt", "ai":"ai",    "authors":"multiple authors", "lang":"british author"},
    
    {"folder":"llmgenerated/SAE_llama", "ai":"ai",    "authors":"single author", "lang":"estonian author"},
    {"folder":"llmgenerated/SAU_llama", "ai":"ai",    "authors":"single author", "lang":"british author"},
    {"folder":"llmgenerated/MAE_llama", "ai":"ai",    "authors":"multiple authors", "lang":"estonian author"},
    {"folder":"llmgenerated/MAU_llama", "ai":"ai",    "authors":"multiple authors", "lang":"british author"}
]
llm_prompt_pairs = [
    {
        "name": "stylometric",
        "model": "microsoft/phi-4",
        "prompt_file": "prompts/stylometric.txt"
    },
    {
        "name": "statistical",
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "prompt_file": "prompts/statistical.txt"
    },
    {
        "name": "argumentation",
        "model": "deepseek/deepseek-r1-0528",
        "prompt_file": "prompts/argumentation.txt"
    },
    {
        "name": "adversarial",#too soft
        "model": "meta-llama/llama-3.3-70b-instruct",
        "prompt_file": "prompts/adversarial.txt"
    },
    {
        "name": "Advocate",#goes of rails
        "model": "qwen/qwen-2.5-72b-instruct",
        "prompt_file": "prompts/advocate.txt"
    }
]

llm_prompt_pairs = [
    {
        "name": "stylometric",
        "model": "openai/gpt-oss-120b",
        "prompt_file": "prompts/stylometric.txt"
    },
    {
        "name": "statistical",
        "model": "openai/gpt-oss-120b",
        "prompt_file": "prompts/statistical.txt"
    },
    {
        "name": "argumentation",
        "model": "openai/gpt-oss-120b",
        "prompt_file": "prompts/argumentation.txt"
    },
    {
        "name": "adversarial",#too soft
        "model": "openai/gpt-oss-120b",
        "prompt_file": "prompts/adversarial.txt"
    },
    {
        "name": "Advocate",#goes of rails
        "model": "openai/gpt-oss-120b",
        "prompt_file": "prompts/advocate.txt"
    }
]

In [ ]:
# pip install openai nest_asyncio

import json
import asyncio
import nest_asyncio
from pathlib import Path
from datetime import datetime
from openai import AsyncOpenAI

nest_asyncio.apply()
semaphore = asyncio.Semaphore(5)  # max 5 korraga

client = AsyncOpenAI(
    api_key= "sk-or-v1-",
    base_url="https://openrouter.ai/api/v1"
)

BASE_PATH = Path(".")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


# -----------------------
# LOAD PROMPTS
# -----------------------

def load_prompt(file):
    with open(file, "r", encoding="utf-8") as f:
        return f.read()


for pair in llm_prompt_pairs:
    pair["prompt_text"] = load_prompt(pair["prompt_file"])


# -----------------------
# BUILD FINAL INPUT
# (PROMPT + META + TEXT)
# -----------------------

def build_input(prompt, meta, text):

    meta_block = f"""
METADATA:
- Authors: {meta['authors']}
- Native language: {meta['lang']}
"""

    return f"""
{prompt}

{meta_block}

TEXT:
{text}
"""


# -----------------------
# CACHE CHECK
# -----------------------

def output_path(file_path, pair_name):
    safe_folder = file_path.parent.name
    return OUTPUT_DIR / f"{safe_folder}__{file_path.stem}__{pair_name}.json"


def exists(file_path, pair_name):
    return output_path(file_path, pair_name).exists()


# -----------------------
# LLM CALL
# -----------------------

async def call_llm(model, prompt):
    async with semaphore:
        try:
            res = await client.chat.completions.create(
                model=model,
                messages=[{"role":"user","content":prompt}],
                temperature=0.7
            )

            return res.choices[0].message.content

        except Exception as e:
            return f"ERROR: {e}"


# -----------------------
# PROCESS ONE (file + pair)
# -----------------------

async def process_one(file_path, meta, pair):

    out_file = output_path(file_path, pair["name"])

    # cache
    if out_file.exists():
        return None

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    full_input = build_input(
        pair["prompt_text"],
        meta,
        text
    )

    output = await call_llm(
        pair["model"],
        full_input
    )

    result = {
        "file": str(file_path),
        "dataset": meta,
        "llm": pair["model"],
        "prompt_name": pair["name"],
        "timestamp": str(datetime.now()),
        "input": full_input,
        "output": output
    }

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return result


# -----------------------
# PARALLEELNE RUN
# -----------------------

async def run_all():

    tasks = []

    for ds in datasets:

        folder = BASE_PATH / ds["folder"]

        if not folder.exists():
            print(f"Missing folder: {folder}")
            continue


        for file in folder.glob("*.txt"):

            for pair in llm_prompt_pairs:

                tasks.append(
                    process_one(file, ds, pair)
                )

    results = await asyncio.gather(*tasks)

    return [r for r in results if r is not None]


# RUN
print("SCRIPT STARTED")
results = asyncio.run(run_all())

print(f"New results: {len(results)}")

In [ ]:
for ds in datasets:
    folder = BASE_PATH / ds["folder"]
    print(ds["folder"], len(list(folder.glob("*.txt"))))

In [ ]:
# pip install openai nest_asyncio

import json
import asyncio
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from openai import AsyncOpenAI

nest_asyncio.apply()

# -----------------------
# CONFIG
# -----------------------

OUTPUT_DIR = Path("outputs")
META_OUTPUT_DIR = Path("meta_outputs")
META_OUTPUT_DIR.mkdir(exist_ok=True)

client = AsyncOpenAI(
    api_key="sk-or-v1-",
    base_url="https://openrouter.ai/api/v1"
)

META_MODEL = "openai/gpt-oss-120b"


# -----------------------
# GROUP FILES
# -----------------------

def group_files():

    groups = defaultdict(list)

    for file in OUTPUT_DIR.glob("*.json"):

        name = file.stem  # ilma .json

        parts = name.split("__")

        if len(parts) < 3:
            continue

        base_key = "__".join(parts[:-1])  # kõik peale viimase (prompt)

        groups[base_key].append(file)

    return groups


# -----------------------
# LOAD JSON
# -----------------------

def load_json(file):

    with open(file, "r", encoding="utf-8") as f:
        return json.load(f)


# -----------------------
# BUILD META PROMPT
# -----------------------

def build_meta_prompt(group_key, items):

    text = f"""
# Role:
You are a Meta-LLM Aggregator for authorship classification.

# Task:
Combine outputs from Stylometric, Statistical, Argumentation, Adversarial, and Human Advocate analysts to produce a final decision.

# Key Principles:
- Integrate all signals; do NOT rely on majority voting
- Prioritize: Argumentation > Human Advocate > Adversarial > Stylometric/Statistical
- Resolve conflicts: reasoning quality is prioritized over surface-level stylistic features.
- If adversarial inconsistencies are strong, consider Uncertain
- If Human Advocate strongly supports human, reduce AI confidence
- Prefer "Uncertain" over incorrect confident classification

# Domain Awareness:
Academic writing may appear formal and uniform; do NOT treat this alone as AI evidence.
Writing can contain OCR and formating errors, ignore those those when considering evidence.

# Output:
Return ONLY JSON:
{{
  "final_prediction": "AI-generated" | "Human-written" | "Uncertain",
  "confidence": 0.0-1.0,
  "evidence_summary": {{
    "ai_signals": ["..."],
    "human_signals": ["..."],
    "conflicts": ["..."]
  }},
  "reasoning": "<2–3 sentence synthesis>"
}}

--- EXPERT OUTPUTS ---
"""

    for item in items:

        text += f"""

EXPERT: {item['prompt_name']}
MODEL: {item['llm']}

OUTPUT:
{item['output']}

-------------------
"""

    return text


# -----------------------
# CALL META MODEL
# -----------------------

semaphore = asyncio.Semaphore(3)

async def call_meta(prompt):

    async with semaphore:

        try:
            res = await client.chat.completions.create(
                model=META_MODEL,
                messages=[{"role":"user","content":prompt}],
                temperature=0.3
            )

            return res.choices[0].message.content

        except Exception as e:
            return f"ERROR: {e}"


# -----------------------
# PROCESS ONE GROUP
# -----------------------

async def process_group(group_key, files):
    if (META_OUTPUT_DIR / f"{group_key}__META.json").exists():
        return None

    items = [load_json(f) for f in files]

    # sanity check (nt 5 eksperti)
    if len(items) < 2:
        print(f"Skipping (too few): {group_key}")
        return None

    meta_info = items[0]["dataset"]

    label = meta_info["ai"]  # "ai" / "human"
    authors = meta_info["authors"]
    lang = meta_info["lang"]

    prompt = build_meta_prompt(group_key, items)

    result = await call_meta(prompt)

    out = {
        "group": group_key,
        "label": label,
        "is_ai": label == "ai",
        "authors": authors,
        "lang": lang,
        "n_experts": len(items),
        "experts": [
            {
                "prompt": i["prompt_name"],
                "model": i["llm"]
            } for i in items
        ],
        "meta_output": result
    }

    out_file = META_OUTPUT_DIR / f"{group_key}__META.json"

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    return out


# -----------------------
# RUN ALL
# -----------------------

async def run_all():

    groups = group_files()

    print(f"Total groups: {len(groups)}")

    tasks = []

    for key, files in groups.items():

        print(f"Group: {key} ({len(files)} experts)")

        tasks.append(
            process_group(key, files)
        )

    results = await asyncio.gather(*tasks)

    return [r for r in results if r is not None]


# RUN
print("Started script")
results = asyncio.run(run_all())

print(f"\nMeta results created: {len(results)}")

In [ ]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd

META_DIR = Path("meta_outputs")

# -----------------------
# LOAD ALL META FILES
# -----------------------

def load_all():

    data = []

    for file in META_DIR.glob("*__META.json"):

        with open(file, "r", encoding="utf-8") as f:
            data.append(json.load(f))

    return data


# -----------------------
# NORMALIZE PREDICTIONS
# -----------------------

def normalize_pred(pred):

    if pred is None:
        return "uncertain"

    pred = pred.lower()

    if "human" in pred:
        return "human"

    if "ai" in pred:
        return "ai"

    if "uncertain" in pred:
        return "uncertain"

    return "uncertain"


def normalize_true(label):

    if label == "ai":
        return "ai"

    if label == "human":
        return "human"

    return "uncertain"


# -----------------------
# BUILD CONFUSION MATRIX
# -----------------------

def build_confusion(data):

    matrix = Counter()

    for item in data:

        true_label = normalize_true(item["label"])

        try:
            pred_raw = json.loads(item["meta_output"])
            pred_label = normalize_pred(pred_raw.get("final_prediction"))

        except:
            pred_label = "uncertain"

        matrix[(true_label, pred_label)] += 1

    return matrix


# -----------------------
# DISPLAY AS TABLE
# -----------------------

def to_dataframe(matrix):

    labels = ["human", "ai", "uncertain"]

    rows = []

    for true in labels:
        row = []
        for pred in labels:
            row.append(matrix[(true, pred)])
        rows.append(row)

    df = pd.DataFrame(
        rows,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels]
    )

    return df


# -----------------------
# RUN
# -----------------------

data = load_all()

matrix = build_confusion(data)

df = to_dataframe(matrix)

print(df)

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd

META_DIR = Path("meta_outputs")


# -----------------------
# SAFE PARSE
# -----------------------

def parse_meta_output(raw):

    try:
        obj = json.loads(raw)
    except:
        return None

    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except:
            return None

    return obj


# -----------------------
# NORMALIZE LABELS
# -----------------------

def normalize_true(x):
    return x  # "ai" / "human"


def normalize_pred(x):

    x = x.lower()

    if "human" in x:
        return "human"

    if "ai" in x:
        return "ai"

    return "uncertain"


# -----------------------
# LOAD DATA
# -----------------------

def load_all():

    data = []

    for f in META_DIR.glob("*__META.json"):

        with open(f, "r", encoding="utf-8") as file:
            data.append(json.load(file))

    return data


# -----------------------
# WEIGHTED CONFUSION MATRIX
# -----------------------

def build_weighted_confusion(data):

    matrix = defaultdict(float)

    for item in data:

        true_label = normalize_true(item["label"])

        parsed = parse_meta_output(item["meta_output"])

        if not parsed:
            continue

        pred_label = normalize_pred(parsed.get("final_prediction", "uncertain"))

        confidence = parsed.get("confidence", 0.0)

        # 🔥 WEIGHTED ADD
        matrix[(true_label, pred_label)] += confidence

    return matrix


# -----------------------
# TO DATAFRAME
# -----------------------

def to_df(matrix):

    labels = ["human", "ai", "uncertain"]

    rows = []

    for t in labels:
        row = []
        for p in labels:
            row.append(round(matrix[(t, p)], 3))
        rows.append(row)

    return pd.DataFrame(
        rows,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels]
    )


# -----------------------
# ACCURACY (WEIGHTED)
# -----------------------

def weighted_accuracy(matrix):

    correct = matrix[("human","human")] + matrix[("ai","ai")]

    total = sum(matrix.values())

    return correct / total if total > 0 else 0


# -----------------------
# RUN
# -----------------------

data = load_all()

matrix = build_weighted_confusion(data)

df = to_df(matrix)

print(df)

print("\nWeighted accuracy:", round(weighted_accuracy(matrix), 4))

In [ ]:
import matplotlib.pyplot as plt

conf = []

for item in data:
    parsed = parse_meta_output(item["meta_output"])
    if parsed:
        conf.append(parsed.get("confidence", 0))

plt.hist(conf)
plt.title("Confidence distribution")
plt.show()

In [ ]:
for item in data:

    parsed = parse_meta_output(item["meta_output"])
    if not parsed:
        continue

    pred = normalize_pred(parsed["final_prediction"])
    true = item["label"]

    if pred != true and parsed.get("confidence", 0) > 0.88:
        print("HIGH CONF ERROR:", item["group"])

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

META_DIR = Path("meta_outputs")


# -----------------------
# SAFE JSON PARSER
# -----------------------
def parse_meta_output(raw):

    if raw is None:
        return None

    try:
        obj = json.loads(raw)
    except:
        return None

    # double-encoded JSON fix
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except:
            return None

    return obj


# -----------------------
# NORMALIZE PREDICTION
# -----------------------
def normalize_pred(pred):

    if pred is None:
        return "uncertain"

    pred = pred.lower()

    if "human" in pred:
        return "human"

    if "ai" in pred:
        return "ai"

    return "uncertain"


# -----------------------
# COLLECT DATA
# -----------------------
xs = []  # confidence
ys = []  # correctness

files = list(META_DIR.glob("*__META.json"))

print("Files found:", len(files))

for f in files:

    with open(f, "r", encoding="utf-8") as file:
        item = json.load(file)

    parsed = parse_meta_output(item.get("meta_output"))

    if not parsed:
        continue

    true_label = item.get("label")
    pred_label = normalize_pred(parsed.get("final_prediction"))

    confidence = parsed.get("confidence", 0)
    if confidence is None:
        confidence = 0

    # correctness (0 / 1)
    correct = 1 if pred_label == true_label else 0

    xs.append(confidence)
    ys.append(correct)


print("Points:", len(xs))


# -----------------------
# PLOT
# -----------------------
plt.figure(figsize=(6,4))
plt.scatter(xs, ys, alpha=0.6)

plt.yticks([0, 1], ["incorrect", "correct"])
plt.xlabel("confidence")
plt.ylabel("correctness")
plt.title("Meta-LLM: Confidence vs Correctness")

plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------
# EXTRACT CONFIDENCE + CORRECTNESS
# -----------------------

def extract_confidence_data(data):

    results = []

    for item in data:

        true_label = normalize_true(item["label"])

        try:
            pred_raw = json.loads(item["meta_output"])
            pred_label = normalize_pred(pred_raw.get("final_prediction"))

            conf = pred_raw.get("confidence", None)

            if conf is None:
                continue

            # normalize confidence to 0–1
            if conf > 1:
                conf = conf / 100.0

            correct = (true_label == pred_label)

            results.append((conf, correct))

        except:
            continue

    return results


# -----------------------
# BINNING (0-10%, 10-20%, ...)
# -----------------------

def compute_bin_accuracy(results):

    bins = np.arange(0, 1.1, 0.1)

    bin_correct = np.zeros(len(bins) - 1)
    bin_total = np.zeros(len(bins) - 1)

    for conf, correct in results:

        idx = np.digitize(conf, bins) - 1

        if 0 <= idx < len(bin_correct):
            bin_total[idx] += 1
            bin_correct[idx] += int(correct)

    accuracy = [
        (bin_correct[i] / bin_total[i]) if bin_total[i] > 0 else 0
        for i in range(len(bin_correct))
    ]

    labels = [f"{int(bins[i]*100)}-{int(bins[i+1]*100)}%" for i in range(len(bins)-1)]

    return labels, accuracy, bin_total


# -----------------------
# PLOT LINE CHART
# -----------------------

def plot_confidence_curve(labels, accuracy):

    plt.figure()

    plt.plot(labels, accuracy, marker="o")

    plt.ylim(0, 1)

    plt.xticks(rotation=45)

    plt.xlabel("Confidence bin")

    plt.ylabel("Accuracy")

    plt.title("Accuracy vs Confidence (10% bins)")

    plt.tight_layout()

    plt.show()


# -----------------------
# RUN
# -----------------------

data = load_all()

results = extract_confidence_data(data)

labels, accuracy, counts = compute_bin_accuracy(results)

plot_confidence_curve(labels, accuracy)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------
# BIN + CALIBRATION DATA
# -----------------------

def compute_reliability(results):

    bins = np.linspace(0, 1, 31)  # 0.0–1.0 in 10% steps

    bin_correct = np.zeros(len(bins) - 1)
    bin_total = np.zeros(len(bins) - 1)
    bin_conf_sum = np.zeros(len(bins) - 1)

    for conf, correct in results:

        idx = np.digitize(conf, bins) - 1

        if 0 <= idx < len(bin_correct):
            bin_total[idx] += 1
            bin_correct[idx] += int(correct)
            bin_conf_sum[idx] += conf

    # avoid division by zero
    avg_conf = [
        (bin_conf_sum[i] / bin_total[i]) if bin_total[i] > 0 else 0
        for i in range(len(bin_total))
    ]

    accuracy = [
        (bin_correct[i] / bin_total[i]) if bin_total[i] > 0 else 0
        for i in range(len(bin_total))
    ]

    return avg_conf, accuracy, bin_total


# -----------------------
# PLOT RELIABILITY DIAGRAM
# -----------------------

def plot_reliability(avg_conf, accuracy, counts):

    plt.figure()

    # model calibration curve
    plt.plot(avg_conf, accuracy, marker="o", label="Model")

    # perfect calibration line
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Reliability Diagram (Calibration Curve)")

    plt.ylim(0, 1)
    plt.xlim(0, 1)

    # optional: show sample sizes
    for i, c in enumerate(counts):
        if c > 0:
            plt.text(avg_conf[i], accuracy[i], str(int(c)), fontsize=9)

    plt.legend()
    plt.tight_layout()
    plt.show()


# -----------------------
# RUN
# -----------------------

data = load_all()
results = extract_confidence_data(data)

avg_conf, accuracy, counts = compute_reliability(results)

plot_reliability(avg_conf, accuracy, counts)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------
# EXTRACT DETAILED OUTCOMES
# -----------------------

def extract_detailed(results_raw):

    data = []

    for item in results_raw:

        true_label = normalize_true(item["label"])

        try:
            pred_raw = json.loads(item["meta_output"])
            pred_label = normalize_pred(pred_raw.get("final_prediction"))

            conf = pred_raw.get("confidence", None)
            if conf is None:
                continue

            if conf > 1:
                conf = conf / 100.0

            # classify outcome types
            if true_label == "ai" and pred_label == "ai":
                outcome = "TP"
            elif true_label == "human" and pred_label == "human":
                outcome = "TN"
            elif true_label == "human" and pred_label == "ai":
                outcome = "FP"
            elif true_label == "ai" and pred_label == "human":
                outcome = "FN"
            else:
                outcome = "UNK"

            data.append((conf, outcome))

        except:
            continue

    return data


# -----------------------
# BINNING
# -----------------------

def compute_fp_fn_bins(data):

    bins = np.linspace(0, 1, 11)

    fp = np.zeros(10)
    fn = np.zeros(10)
    tp = np.zeros(10)
    tn = np.zeros(10)

    for conf, outcome in data:

        idx = np.digitize(conf, bins) - 1

        if 0 <= idx < 10:

            if outcome == "FP":
                fp[idx] += 1
            elif outcome == "FN":
                fn[idx] += 1
            elif outcome == "TP":
                tp[idx] += 1
            elif outcome == "TN":
                tn[idx] += 1

    return bins, tp, tn, fp, fn


# -----------------------
# PLOT
# -----------------------

def plot_fp_fn(bins, tp, tn, fp, fn):

    labels = [f"{int(bins[i]*100)}-{int(bins[i+1]*100)}%" for i in range(len(bins)-1)]
    x = np.arange(len(labels))

    plt.figure()

    plt.plot(x, fp, marker="o", label="False Positives (AI predicted, actually human)")
    plt.plot(x, fn, marker="o", label="False Negatives (Human predicted, actually AI)")
    plt.plot(x, tp, marker="o", label="True Positives")
    plt.plot(x, tn, marker="o", label="True Negatives")

    plt.xticks(x, labels, rotation=45)
    plt.xlabel("Confidence bin")
    plt.ylabel("Count")
    plt.title("Error Types vs Confidence")

    plt.legend()
    plt.tight_layout()
    plt.show()


# -----------------------
# RUN
# -----------------------

data = load_all()

detailed = extract_detailed(data)

bins, tp, tn, fp, fn = compute_fp_fn_bins(detailed)

plot_fp_fn(bins, tp, tn, fp, fn)

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

META_DIR = Path("meta_outputs")


# -----------------------
# LOAD
# -----------------------

def load_all():
    data = []
    for file in META_DIR.glob("*__META.json"):
        with open(file, "r", encoding="utf-8") as f:
            data.append(json.load(f))
    return data


# -----------------------
# NORMALIZE
# -----------------------

def normalize_pred(pred):
    if pred is None:
        return "uncertain"
    pred = pred.lower()
    if "human" in pred:
        return "human"
    if "ai" in pred:
        return "ai"
    return "uncertain"


def normalize_true(label):
    if label == "ai":
        return "ai"
    if label == "human":
        return "human"
    return "uncertain"


# -----------------------
# BALANCED FP / FN PROBABILITY
# -----------------------

def compute_balanced_fp_fn(data):

    counts = {
        "human": 0,
        "ai": 0
    }

    errors = {
        "FP": 0,  # human → ai
        "FN": 0   # ai → human
    }

    for item in data:

        true_label = normalize_true(item["label"])

        try:
            pred_raw = json.loads(item["meta_output"])
            pred_label = normalize_pred(pred_raw.get("final_prediction"))
        except:
            continue

        if true_label == "uncertain" or pred_label == "uncertain":
            continue

        # count classes
        counts[true_label] += 1

        # errors
        if true_label == "human" and pred_label == "ai":
            errors["FP"] += 1

        if true_label == "ai" and pred_label == "human":
            errors["FN"] += 1

    # -----------------------
    # BALANCING
    # -----------------------

    # target: equal class weight
    total = counts["human"] + counts["ai"]
    if total == 0:
        return None

    target = total / 2

    # scaling factors
    human_w = target / counts["human"] if counts["human"] > 0 else 1
    ai_w = target / counts["ai"] if counts["ai"] > 0 else 1

    # weighted probabilities
    fp_prob = (errors["FP"] * human_w) / target
    fn_prob = (errors["FN"] * ai_w) / target

    return {
        "FPR_balanced": fp_prob,
        "FNR_balanced": fn_prob,
        "human_weight": human_w,
        "ai_weight": ai_w,
        "counts": counts,
        "errors": errors
    }


# -----------------------
# RUN
# -----------------------

data = load_all()

result = compute_balanced_fp_fn(data)

print(result)

In [ ]:
# pip install openai nest_asyncio

import json
import asyncio
import nest_asyncio
from pathlib import Path
from datetime import datetime
from openai import AsyncOpenAI

nest_asyncio.apply()
semaphore = asyncio.Semaphore(5)  # max 5 korraga

client = AsyncOpenAI(
    api_key= "sk-or-v1-",
    base_url="https://openrouter.ai/api/v1"
)

BASE_PATH = Path(".")
OUTPUT_DIR = Path("zeroshot_baseline_outputs_nemotron")
OUTPUT_DIR.mkdir(exist_ok=True)


# -----------------------
# LOAD PROMPTS
# -----------------------
pair={
        "name": "zeroshotbaselinenemotron",
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "prompt_file": "prompts/zeroshot_baseline.txt"
        }

def load_prompt(file):
    with open(file, "r", encoding="utf-8") as f:
        return f.read()

pair["prompt_text"] = load_prompt(pair["prompt_file"])


# -----------------------
# BUILD FINAL INPUT
# (PROMPT + META + TEXT)
# -----------------------

def build_input(prompt, meta, text):

    meta_block = f"""
METADATA:
- Authors: {meta['authors']}
- Native language: {meta['lang']}
"""

    return f"""
{prompt}

{meta_block}

TEXT:
{text}
"""


# -----------------------
# CACHE CHECK
# -----------------------

def output_path(file_path, pair_name):
    safe_folder = file_path.parent.name
    return OUTPUT_DIR / f"{safe_folder}__{file_path.stem}__{pair_name}.json"


def exists(file_path, pair_name):
    return output_path(file_path, pair_name).exists()


# -----------------------
# LLM CALL
# -----------------------

async def call_llm(model, prompt):
    async with semaphore:
        try:
            res = await client.chat.completions.create(
                model=model,
                messages=[{"role":"user","content":prompt}],
                temperature=0.7
            )

            return res.choices[0].message.content

        except Exception as e:
            return f"ERROR: {e}"


# -----------------------
# PROCESS ONE (file + pair)
# -----------------------

async def process_one(file_path, meta, pair):

    out_file = output_path(file_path, pair["name"])

    # cache
    if out_file.exists():
        return None

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    full_input = build_input(
        pair["prompt_text"],
        meta,
        text
    )

    output = await call_llm(
        pair["model"],
        full_input
    )

    result = {
        "file": str(file_path),
        "dataset": meta,
        "llm": pair["model"],
        "prompt_name": pair["name"],
        "timestamp": str(datetime.now()),
        "input": full_input,
        "output": output
    }

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return result


# -----------------------
# PARALLEELNE RUN
# -----------------------

async def run_all():

    tasks = []

    for ds in datasets:

        folder = BASE_PATH / ds["folder"]

        if not folder.exists():
            print(f"Missing folder: {folder}")
            continue


        for file in folder.glob("*.txt"):
            tasks.append(
                process_one(file, ds, pair)
            )

    results = await asyncio.gather(*tasks)

    return [r for r in results if r is not None]


# RUN
print("SCRIPT STARTED")
results = asyncio.run(run_all())

print(f"New results: {len(results)}")

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

META_DIR = Path("zeroshot_outputs")


# -----------------------
# LOAD
# -----------------------

def load_all():
    data = []
    for file in META_DIR.glob("*__zeroshot.json"):
        with open(file, "r", encoding="utf-8") as f:
            data.append(json.load(f))
    return data


# -----------------------
# NORMALIZE
# -----------------------

def normalize_pred(pred):
    if pred is None:
        return "uncertain"
    pred = pred.lower()
    if "human" in pred:
        return "human"
    if "ai" in pred:
        return "ai"
    return "uncertain"


def normalize_true(label):
    if label == "ai":
        return "ai"
    if label == "human":
        return "human"
    return "uncertain"


# -----------------------
# BALANCED FP / FN PROBABILITY
# -----------------------

def compute_balanced_fp_fn(data):

    counts = {
        "human": 0,
        "ai": 0
    }

    errors = {
        "FP": 0,  # human → ai
        "FN": 0   # ai → human
    }

    for item in data:

        true_label = normalize_true(item["dataset"]["ai"])

        try:
            pred_raw = json.loads(item["output"])
            pred_label = normalize_pred(pred_raw.get("final_prediction"))
        except:
            continue

        if true_label == "uncertain" or pred_label == "uncertain":
            continue

        # count classes
        counts[true_label] += 1

        # errors
        if true_label == "human" and pred_label == "ai":
            errors["FP"] += 1

        if true_label == "ai" and pred_label == "human":
            errors["FN"] += 1

    # -----------------------
    # BALANCING
    # -----------------------

    # target: equal class weight
    total = counts["human"] + counts["ai"]
    if total == 0:
        return None

    target = total / 2

    # scaling factors
    human_w = target / counts["human"] if counts["human"] > 0 else 1
    ai_w = target / counts["ai"] if counts["ai"] > 0 else 1

    # weighted probabilities
    fp_prob = (errors["FP"] * human_w) / target
    fn_prob = (errors["FN"] * ai_w) / target

    return {
        "FPR_balanced": fp_prob,
        "FNR_balanced": fn_prob,
        "human_weight": human_w,
        "ai_weight": ai_w,
        "counts": counts,
        "errors": errors
    }


# -----------------------
# RUN
# -----------------------

data = load_all()

result = compute_balanced_fp_fn(data)

print(result)

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd

META_DIR = Path("zeroshot_outputs")


# -----------------------
# SAFE PARSE
# -----------------------

def parse_meta_output(raw):

    try:
        obj = json.loads(raw)
    except:
        return None

    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except:
            return None

    return obj


# -----------------------
# NORMALIZE LABELS
# -----------------------

def normalize_true(x):
    return x  # "ai" / "human"


def normalize_pred(x):

    x = x.lower()

    if "human" in x:
        return "human"

    if "ai" in x:
        return "ai"

    return "uncertain"


# -----------------------
# LOAD DATA
# -----------------------

def load_all():

    data = []

    for f in META_DIR.glob("*__zeroshot.json"):

        with open(f, "r", encoding="utf-8") as file:
            data.append(json.load(file))

    return data


# -----------------------
# WEIGHTED CONFUSION MATRIX
# -----------------------

def build_weighted_confusion(data):

    matrix = defaultdict(float)

    for item in data:

        true_label = normalize_true(item["dataset"]["ai"])

        parsed = parse_meta_output(item["output"])

        if not parsed:
            continue

        pred_label = normalize_pred(parsed.get("final_prediction", "uncertain"))

        confidence = parsed.get("confidence", 0.0)

        # 🔥 WEIGHTED ADD
        matrix[(true_label, pred_label)] += confidence

    return matrix


# -----------------------
# TO DATAFRAME
# -----------------------

def to_df(matrix):

    labels = ["human", "ai", "uncertain"]

    rows = []

    for t in labels:
        row = []
        for p in labels:
            row.append(round(matrix[(t, p)], 3))
        rows.append(row)

    return pd.DataFrame(
        rows,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels]
    )


# -----------------------
# ACCURACY (WEIGHTED)
# -----------------------

def weighted_accuracy(matrix):

    correct = matrix[("human","human")] + matrix[("ai","ai")]

    total = sum(matrix.values())

    return correct / total if total > 0 else 0


# -----------------------
# RUN
# -----------------------

data = load_all()

matrix = build_weighted_confusion(data)

df = to_df(matrix)

print(df)

print("\nWeighted accuracy:", round(weighted_accuracy(matrix), 4))

In [ ]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd

META_DIR = Path("zeroshot_baseline_outputs_phi")

# -----------------------
# LOAD ALL META FILES
# -----------------------

def load_all():

    data = []

    for file in META_DIR.glob("*__zeroshotbaselinephi.json"):

        with open(file, "r", encoding="utf-8") as f:
            data.append(json.load(f))

    return data


# -----------------------
# NORMALIZE PREDICTIONS
# -----------------------

def normalize_pred(pred):

    if pred is None:
        return "uncertain"

    pred = pred.strip().lower()

    # exact or strong match first
    if pred in ["uncertain", "unknown"]:
        return "uncertain"

    if pred == "human":
        return "human"

    if pred == "ai":
        return "ai"

    # fallback (safer logic)
    if pred.startswith("human"):
        return "human"

    if pred.startswith("ai"):
        return "ai"

    return "uncertain"


def normalize_true(label):

    if label == "ai":
        return "ai"

    if label == "human":
        return "human"

    return "uncertain"


# -----------------------
# BUILD CONFUSION MATRIX
# -----------------------

import json
import re
from collections import Counter


def extract_json(text):
    """
    Extract JSON object from model output.
    Handles markdown fenced JSON and noisy outputs.
    """

    if not isinstance(text, str):
        return None

    # ```json ... ```
    fenced_match = re.search(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        text,
        re.DOTALL,
    )

    if fenced_match:
        return fenced_match.group(1)

    # Raw {...}
    direct_match = re.search(
        r"(\{.*\})",
        text,
        re.DOTALL,
    )

    if direct_match:
        return direct_match.group(1)

    return None


def extract_prediction_regex(text):
    """
    Fallback extraction if JSON parsing fails.
    """

    if not isinstance(text, str):
        return "uncertain"

    match = re.search(
        r'"?final_prediction"?\s*:\s*"?(AI-generated|Human-written|Uncertain)"?',
        text,
        re.IGNORECASE,
    )

    if match:
        return normalize_pred(match.group(1))

    return "uncertain"


def build_confusion(data):

    matrix = Counter()

    for item in data:

        true_label = normalize_true(item["dataset"]["ai"])

        output = item.get("output", "")

        try:
            # First attempt: direct JSON parse
            pred_raw = json.loads(output)

        except Exception:

            try:
                # Second attempt: extract JSON block
                extracted = extract_json(output)

                if extracted is None:
                    raise ValueError("No JSON found")

                pred_raw = json.loads(extracted)

            except Exception:
                # Final fallback: regex extraction
                pred_label = extract_prediction_regex(output)

            else:
                pred_label = normalize_pred(
                    pred_raw.get("final_prediction")
                )

        else:
            pred_label = normalize_pred(
                pred_raw.get("final_prediction")
            )

        matrix[(true_label, pred_label)] += 1

    return matrix


# -----------------------
# DISPLAY AS TABLE
# -----------------------

def to_dataframe(matrix):

    labels = ["human", "ai", "uncertain"]

    rows = []

    for true in labels:
        row = []
        for pred in labels:
            row.append(matrix[(true, pred)])
        rows.append(row)

    df = pd.DataFrame(
        rows,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels]
    )

    return df


# -----------------------
# RUN
# -----------------------

data = load_all()

matrix = build_confusion(data)

df = to_dataframe(matrix)

print(df)

In [ ]:
!pip install statsmodels
import json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

# =========================================================
# CONFIG
# =========================================================

ZEROSHOT_DIR = Path("zeroshot_outputs")
MULTISTAGE_DIR = Path("meta_outputs_v2")

# =========================================================
# EXTRACT TRUE + PRED LABEL
# =========================================================

def extract_labels(item):

    # -------------------------
    # TRUE LABEL
    # -------------------------

    if "dataset" in item and "ai" in item["dataset"]:
        true_raw = item["dataset"]["ai"]

    elif "label" in item:
        true_raw = item["label"]

    else:
        true_raw = "uncertain"

    true_label = normalize_true(true_raw)

    # -------------------------
    # PREDICTION FIELD
    # -------------------------

    pred_container = None

    if "output" in item:
        pred_container = item["output"]

    elif "meta_output" in item:
        pred_container = item["meta_output"]

    # -------------------------
    # PARSE PREDICTION
    # -------------------------

    try:

        pred_raw = json.loads(pred_container)

        pred_label = normalize_pred(
            pred_raw.get("final_prediction")
        )

    except Exception:
        pred_label = "uncertain"

    return true_label, pred_label
    
# =========================================================
# LOAD FILES
# =========================================================

def load_all(directory, pattern):

    data = {}

    for file in directory.glob(pattern):

        with open(file, "r", encoding="utf-8") as f:
            item = json.load(f)

        # normalize filename key
        sample_id = (
            file.stem
            .replace("__zeroshot", "")
            .replace("__META", "")
        )

        data[sample_id] = item

    return data

# =========================================================
# NORMALIZATION
# =========================================================

def normalize_pred(pred):

    if pred is None:
        return "uncertain"

    pred = str(pred).strip().lower()

    if pred in ["uncertain", "unknown"]:
        return "uncertain"

    if pred == "human":
        return "human"

    if pred == "ai":
        return "ai"

    if pred.startswith("human"):
        return "human"

    if pred.startswith("ai"):
        return "ai"

    return "uncertain"


def normalize_true(label):

    if label == "ai":
        return "ai"

    if label == "human":
        return "human"

    return "uncertain"


# =========================================================
# BUILD CONFUSION MATRIX
# =========================================================

def build_confusion(data):

    matrix = Counter()

    for item in data.values():

        true_label, pred_label = extract_labels(item)

        matrix[(true_label, pred_label)] += 1

    return matrix


def to_dataframe(matrix):

    labels = ["human", "ai", "uncertain"]

    rows = []

    for true in labels:

        row = []

        for pred in labels:
            row.append(matrix[(true, pred)])

        rows.append(row)

    return pd.DataFrame(
        rows,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels]
    )


# =========================================================
# ACCURACY
# =========================================================

def compute_accuracy(data):

    correct = 0
    total = 0

    for item in data.values():

        true_label, pred_label = extract_labels(item)

        if true_label == pred_label:
            correct += 1

        total += 1

    return correct / total


# =========================================================
# MCNEMAR TEST
# =========================================================

def run_mcnemar(zeroshot_data, multistage_data):

    shared_ids = sorted(
        set(zeroshot_data.keys()) &
        set(multistage_data.keys())
    )

    n11 = 0  # both correct
    n10 = 0  # zeroshot correct, multistage wrong
    n01 = 0  # zeroshot wrong, multistage correct
    n00 = 0  # both wrong

    for sample_id in shared_ids:
        

        z_true, z_pred = extract_labels(
            zeroshot_data[sample_id]
        )

        m_true, m_pred = extract_labels(
            multistage_data[sample_id]
        )

        z_correct = z_true == z_pred
        m_correct = m_true == m_pred

        if z_correct != m_correct:
            print(sample_id, z_true, z_pred, m_pred)

        # sanity check
        if z_true != m_true:
            print(f"WARNING: label mismatch for {sample_id}")
            continue

        z_correct = (z_true == z_pred)
        m_correct = (m_true == m_pred)

        if z_correct and m_correct:
            n11 += 1

        elif z_correct and not m_correct:
            n10 += 1

        elif not z_correct and m_correct:
            n01 += 1

        else:
            n00 += 1

    table = [
        [n11, n10],
        [n01, n00]
    ]

    result = mcnemar(table, exact=True)

    return {
        "table": table,
        "statistic": result.statistic,
        "pvalue": result.pvalue,
        "n11": n11,
        "n10": n10,
        "n01": n01,
        "n00": n00,
    }


# =========================================================
# RUN
# =========================================================

zeroshot_data = load_all(
    ZEROSHOT_DIR,
    "*__zeroshot.json"
)

multistage_data = load_all(
    MULTISTAGE_DIR,
    "*.json"
)

# -------------------------
# Confusion matrices
# -------------------------

zeroshot_matrix = build_confusion(zeroshot_data)
multistage_matrix = build_confusion(multistage_data)

print("\nZEROSHOT CONFUSION MATRIX\n")
print(to_dataframe(zeroshot_matrix))

print("\nMULTISTAGE CONFUSION MATRIX\n")
print(to_dataframe(multistage_matrix))

# -------------------------
# Accuracy
# -------------------------

zeroshot_acc = compute_accuracy(zeroshot_data)
multistage_acc = compute_accuracy(multistage_data)

print("\nACCURACY\n")

print(f"Zeroshot accuracy:   {zeroshot_acc:.4f}")
print(f"Multistage accuracy: {multistage_acc:.4f}")

# -------------------------
# McNemar test
# -------------------------

results = run_mcnemar(
    zeroshot_data,
    multistage_data
)

print("\nMCNEMAR TEST\n")

print("Contingency table:")
print(np.array(results["table"]))

print(f"\nBoth correct:              {results['n11']}")
print(f"Zeroshot only correct:     {results['n10']}")
print(f"Multistage only correct:   {results['n01']}")
print(f"Both wrong:                {results['n00']}")

print(f"\nStatistic: {results['statistic']}")
print(f"P-value:   {results['pvalue']:.6f}")

# -------------------------
# Interpretation
# -------------------------

alpha = 0.05

print("\nINTERPRETATION\n")

if results["pvalue"] < alpha:
    print(
        "The difference between models is "
        "STATISTICALLY SIGNIFICANT "
        f"(p < {alpha})."
    )
else:
    print(
        "No statistically significant difference "
        f"was found (p >= {alpha})."
    )

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

# =========================================================
# CONFIG
# =========================================================

ZEROSHOT_DIR = Path("zeroshot_outputs")
MULTISTAGE_DIR = Path("meta_outputs")


# =========================================================
# LOAD FILES
# =========================================================

def load_all(directory, pattern):

    data = {}

    for file in directory.glob(pattern):

        with open(file, "r", encoding="utf-8") as f:
            item = json.load(f)

        sample_id = file.stem
        data[sample_id] = item

    return data


# =========================================================
# NORMALIZATION
# =========================================================

def normalize_pred(pred):

    if pred is None:
        return "uncertain"

    pred = str(pred).strip().lower()

    if pred in ["uncertain", "unknown"]:
        return "uncertain"

    if pred.startswith("human"):
        return "human"

    if pred.startswith("ai"):
        return "ai"

    return "uncertain"


def normalize_true(label):

    if label == "ai":
        return "ai"
    if label == "human":
        return "human"
    return "uncertain"


# =========================================================
# EXTRACT LABELS
# =========================================================

def extract_labels(item):

    if "dataset" in item and "ai" in item["dataset"]:
        true_raw = item["dataset"]["ai"]
    elif "label" in item:
        true_raw = item["label"]
    else:
        true_raw = "uncertain"

    true_label = normalize_true(true_raw)

    pred_container = None

    if "output" in item:
        pred_container = item["output"]
    elif "meta_output" in item:
        pred_container = item["meta_output"]

    try:
        pred_raw = json.loads(pred_container)
        pred_label = normalize_pred(pred_raw.get("final_prediction"))
    except:
        pred_label = "uncertain"

    return true_label, pred_label


def is_correct(item):

    true, pred = extract_labels(item)
    return int(true == pred)


# =========================================================
# PARSE METADATA
# =========================================================

import re

def parse_metadata(sample_id: str):

    parts = sample_id.split("__")

    # -----------------------------
    # AUTHOR (always first chunk)
    # -----------------------------
    author = parts[0]

    # -----------------------------
    # MODEL detection (robust)
    # -----------------------------
    known_models = ["gpt", "llama", "gemini", "deepseek"]

    model = "baseline"

    for p in parts:
        p_low = p.lower()
        for m in known_models:
            if m in p_low:
                model = m
                break

    # -----------------------------
    # BASE ID extraction
    # -----------------------------

    # remove known prefixes + suffixes
    base = sample_id

    # remove author + model parts
    for p in parts:
        if p == author:
            base = base.replace(p + "__", "")

    # remove model tokens
    for m in known_models:
        base = re.sub(rf"__?{m}__?", "", base, flags=re.I)

    # remove known suffixes
    base = base.replace("__META", "")
    base = re.sub(r"_masked.*", "", base)

    return author, model, base


# =========================================================
# BUILD INDEX
# =========================================================

def build_index(data):

    index = {}

    for k, v in data.items():
        author, model, base = parse_metadata(k)
        index[(author, model, base)] = v

    return index


# =========================================================
# GROUPED ANALYSIS
# =========================================================

def evaluate_grouped(zero_idx, multi_idx):

    results = []

    all_keys = set(zero_idx.keys()) | set(multi_idx.keys())

    grouped = defaultdict(lambda: {"z": [], "m": []})

    # -------------------------
    # assign samples
    # -------------------------
    for k in all_keys:

        author, model, base = k

        if k in zero_idx:
            grouped[(author, model)]["z"].append((base, zero_idx[k]))

        if k in multi_idx:
            grouped[(author, model)]["m"].append((base, multi_idx[k]))

    # -------------------------
    # evaluate each group
    # -------------------------
    for (author, model), vals in grouped.items():

        z_items = dict(vals["z"])
        m_items = dict(vals["m"])

        # accuracy
        z_acc = np.mean([is_correct(v) for v in z_items.values()]) if z_items else None
        m_acc = np.mean([is_correct(v) for v in m_items.values()]) if m_items else None

        # McNemar (paired on base id)
        shared = set(z_items.keys()) & set(m_items.keys())

        n11 = n10 = n01 = n00 = 0

        for b in shared:

            z_c = is_correct(z_items[b])
            m_c = is_correct(m_items[b])

            if z_c and m_c:
                n11 += 1
            elif z_c and not m_c:
                n10 += 1
            elif not z_c and m_c:
                n01 += 1
            else:
                n00 += 1

        p_value = None

        if (n10 + n01) > 0:
            table = [[n11, n10], [n01, n00]]
            p_value = mcnemar(table, exact=True).pvalue

        results.append({
            "author": author,
            "model": model,
            "zeroshot_acc": z_acc,
            "multistage_acc": m_acc,
            "delta": None if z_acc is None or m_acc is None else m_acc - z_acc,
            "n_pairs": len(shared),
            "n10": n10,
            "n01": n01,
            "p_value": p_value
        })

    return pd.DataFrame(results)


# =========================================================
# RUN
# =========================================================

zeroshot_data = load_all(ZEROSHOT_DIR, "*__zeroshot.json")
multistage_data = load_all(MULTISTAGE_DIR, "*.json")

zero_idx = build_index(zeroshot_data)
multi_idx = build_index(multistage_data)

df = evaluate_grouped(zero_idx, multi_idx)

# =========================================================
# OUTPUT
# =========================================================

df = df.sort_values(["author", "model"])

print("\n=== GROUPED RESULTS ===\n")
print(df)

df.to_csv("grouped_model_comparison.csv", index=False)

print("\nSaved to grouped_model_comparison.csv")